# PPO Attack Environment
This notebook validates the detector-agnostic environment with a synthetic RGB image and `MockDetector`. No external data or model weights are downloaded.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from stable_baselines3.common.env_checker import check_env

from src.attacks.perturbations import ACTIONS, apply_action
from src.ppo.environment import DeepfakeAttackEnv
from src.ppo.mock_detector import MockDetector
from src.ppo.reward import confidence_reduction_reward, quality_aware_reward

## 2. Create one synthetic test image

In [ ]:
height, width = 160, 224
x = np.linspace(48, 240, width, dtype=np.uint8)
gradient = np.tile(x, (height, 1))
checker = ((np.indices((height, width)).sum(axis=0) // 12) % 2 * 25).astype(np.uint8)
pixels = np.stack([gradient, np.clip(gradient + checker, 0, 255), gradient], axis=2)
image = Image.fromarray(pixels).convert('RGB')
display(image)

## 3. Visualize every perturbation

In [ ]:
rng = np.random.default_rng(42)
outputs = {name: apply_action(image, action_id, rng=rng) for action_id, name in ACTIONS.items()}
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for axis, (name, output) in zip(axes.flat, outputs.items()):
    axis.imshow(output)
    axis.set_title(name.replace('_', ' ').title())
    axis.axis('off')
axes.flat[-1].axis('off')
plt.tight_layout()

## 4. Validate the action space

In [ ]:
for action_id, action_name in ACTIONS.items():
    output = apply_action(image, action_id, rng=np.random.default_rng(42))
    assert output.mode == 'RGB' and output.size == image.size
    print(action_id, action_name, output.mode, output.size)

## 5. Reward examples

In [ ]:
examples = [(0.9, 0.5), (0.5, 0.9), (0.7, 0.7)]
for before, after in examples:
    print(before, after, confidence_reduction_reward(before, after))
print('quality-aware:', quality_aware_reward(0.9, 0.5, distortion=0.2))

## 6. Confirm that the mock detector responds to image changes

In [ ]:
detector = MockDetector()
for action_id, action_name in ACTIONS.items():
    attacked = apply_action(image, action_id, rng=np.random.default_rng(42))
    print(f'{action_name:16s} P(fake)={detector.predict(attacked):.4f}')

## 7. Exercise `reset()` and `step()`

In [ ]:
env = DeepfakeAttackEnv(image, detector, max_steps=5, seed=42)
obs, info = env.reset()
print('reset:', obs, info)
for action in [1, 2, 5]:
    obs, reward, terminated, truncated, info = env.step(action)
    print('step:', obs, reward, terminated, truncated, info)
    if terminated or truncated:
        break

## 8. Run the Stable-Baselines3 environment checker

In [ ]:
check_env(DeepfakeAttackEnv(image, detector, seed=42), warn=True)
print('Environment check passed.')